# 2. Veri Ön İşleme ve Birleştirme

Bu notebook'ta verileri temizleyecek, yıldırım olaylarını en yakın istasyona eşleştirip tüm verileri birleştireceğiz.

**İçerik:**
- Önceki verilerin yüklenmesi
- Koordinat bazlı istasyon eşleştirmesi
- Günlük agregasyon
- Verilerin birleştirilmesi

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import warnings
import os

warnings.filterwarnings('ignore')
print('Kütüphaneler yüklendi!')

## 2.1 Verilerin Yüklenmesi

In [ ]:
# Önceki notebook'tan kaydedilen verileri yükle
DATA_PATH = '../data/'

yildirim_df = pd.read_pickle(DATA_PATH + 'yildirim_processed.pkl')
istasyonlar = pd.read_pickle(DATA_PATH + 'istasyonlar.pkl')

print(f"Yıldırım verisi: {len(yildirim_df):,} kayıt")
print(f"İstasyon sayısı: {len(istasyonlar)}")

## 2.2 En Yakın İstasyon Eşleştirmesi

In [ ]:
def en_yakin_istasyon(enlem, boylam, istasyonlar_df):
    """Verilen koordinata en yakın istasyonu bulur."""
    ist_coords = istasyonlar_df[['enlem', 'boylam']].values
    nokta = np.array([[enlem, boylam]])
    mesafeler = cdist(nokta, ist_coords, metric='euclidean')[0]
    en_yakin_idx = np.argmin(mesafeler)
    return istasyonlar_df.iloc[en_yakin_idx]['istasyon_no']

# Her yıldırım için en yakın istasyonu bul
print("En yakın istasyon eşleştirmesi yapılıyor...")
yildirim_df['istasyon_no'] = yildirim_df.apply(
    lambda row: en_yakin_istasyon(row['enlem'], row['boylam'], istasyonlar), axis=1
)
print("Eşleştirme tamamlandı!")

# İstasyon başına dağılım
print("\nİstasyon başına yıldırım dağılımı:")
print(yildirim_df['istasyon_no'].value_counts())

## 2.3 Günlük Agregasyon (İstasyon Bazında)

In [ ]:
# İstasyon ve tarih bazında günlük yıldırım sayısı
gunluk_ist_yildirim = yildirim_df.groupby(['tarih', 'istasyon_no']).agg({
    'akim_kA': ['count', 'mean', 'max', 'min'],
    'mesafe_km': 'mean'
}).reset_index()

# Sütun isimlerini düzelt
gunluk_ist_yildirim.columns = ['tarih', 'istasyon_no', 'yildirim_sayisi', 
                                'ort_akim', 'maks_akim', 'min_akim', 'ort_mesafe']

gunluk_ist_yildirim['tarih'] = pd.to_datetime(gunluk_ist_yildirim['tarih'])

print(f"Günlük istasyon bazlı kayıt sayısı: {len(gunluk_ist_yildirim):,}")
display(gunluk_ist_yildirim.head(10))

## 2.4 Meteorolojik Verilerin Düzenlenmesi

In [ ]:
def duzenle_meteo(dosya_adi):
    """Meteorolojik veriyi long format'a çevirir."""
    try:
        df = pd.read_pickle(DATA_PATH + dosya_adi)
        
        # İlk sütun tarih, diğerleri istasyonlar olmalı
        # Sütun yapısını incele
        print(f"\n{dosya_adi} sütunları: {df.columns.tolist()[:5]}...")
        
        return df
    except Exception as e:
        print(f"Hata ({dosya_adi}): {e}")
        return None

# Meteorolojik verileri yükle
meteo_files = [f for f in os.listdir(DATA_PATH) if f.startswith('meteo_')]
print("Meteorolojik veri dosyaları:")
for f in meteo_files:
    print(f"  - {f}")

In [ ]:
# Her meteorolojik veriyi yükle ve incele
meteo_dict = {}
for f in meteo_files:
    veri_adi = f.replace('meteo_', '').replace('.pkl', '')
    meteo_dict[veri_adi] = pd.read_pickle(DATA_PATH + f)
    print(f"{veri_adi}: {meteo_dict[veri_adi].shape}")

In [ ]:
# Örnek bir meteorolojik veri yapısını incele
ornek_veri = list(meteo_dict.values())[0]
print("Örnek veri yapısı:")
display(ornek_veri.head())
print(f"\nSütunlar: {ornek_veri.columns.tolist()}")

In [ ]:
def melted_meteo(df, deger_adi):
    """Meteorolojik veriyi long format'a çevirir."""
    # İlk sütunun tarih olduğunu varsayalım
    tarih_sutun = df.columns[0]
    
    # Melt işlemi
    df_melted = df.melt(id_vars=[tarih_sutun], var_name='istasyon_info', value_name=deger_adi)
    
    # Tarih sütununu datetime'a çevir
    df_melted['tarih'] = pd.to_datetime(df_melted[tarih_sutun], errors='coerce')
    
    return df_melted[['tarih', 'istasyon_info', deger_adi]]

# Test
test_melted = melted_meteo(meteo_dict['maks_sicaklik'], 'maks_sicaklik')
print("Melted veri:")
display(test_melted.head(10))

## 2.5 Tam Tarih Aralığı Oluşturma

In [ ]:
# Veri setinin tarih aralığını belirle
min_tarih = yildirim_df['tarih'].min()
max_tarih = yildirim_df['tarih'].max()

print(f"Tarih aralığı: {min_tarih} - {max_tarih}")

# Tüm tarihler için DataFrame oluştur
tum_tarihler = pd.date_range(start=min_tarih, end=max_tarih, freq='D')
print(f"Toplam gün sayısı: {len(tum_tarihler)}")

# Her istasyon için tüm tarihleri içeren DataFrame
from itertools import product

tum_kombinasyonlar = pd.DataFrame(
    list(product(tum_tarihler, istasyonlar['istasyon_no'])),
    columns=['tarih', 'istasyon_no']
)

print(f"Toplam kombinasyon: {len(tum_kombinasyonlar):,}")

In [ ]:
# Yıldırım verisiyle birleştir
ana_df = tum_kombinasyonlar.merge(
    gunluk_ist_yildirim,
    on=['tarih', 'istasyon_no'],
    how='left'
)

# NaN değerleri 0 ile doldur (yıldırım olmayan günler)
ana_df['yildirim_sayisi'] = ana_df['yildirim_sayisi'].fillna(0).astype(int)

# Hedef değişken: Yıldırım var mı yok mu (binary)
ana_df['yildirim_var'] = (ana_df['yildirim_sayisi'] > 0).astype(int)

print(f"Ana veri seti boyutu: {ana_df.shape}")
print(f"\nHedef değişken dağılımı:")
print(ana_df['yildirim_var'].value_counts())
print(f"\nYıldırım olan gün oranı: {ana_df['yildirim_var'].mean()*100:.2f}%")

## 2.6 Zaman Özellikleri Ekleme

In [ ]:
# Zaman özellikleri
ana_df['yil'] = ana_df['tarih'].dt.year
ana_df['ay'] = ana_df['tarih'].dt.month
ana_df['gun'] = ana_df['tarih'].dt.day
ana_df['haftanin_gunu'] = ana_df['tarih'].dt.dayofweek
ana_df['yilin_gunu'] = ana_df['tarih'].dt.dayofyear

# Mevsim
def mevsim_bul(ay):
    if ay in [12, 1, 2]:
        return 0  # Kış
    elif ay in [3, 4, 5]:
        return 1  # İlkbahar
    elif ay in [6, 7, 8]:
        return 2  # Yaz
    else:
        return 3  # Sonbahar

ana_df['mevsim'] = ana_df['ay'].apply(mevsim_bul)

print("Zaman özellikleri eklendi!")
display(ana_df.head())

## 2.7 Verilerin Kaydedilmesi

In [ ]:
# Ana veri setini kaydet
ana_df.to_pickle(DATA_PATH + 'ana_veri_seti.pkl')
gunluk_ist_yildirim.to_pickle(DATA_PATH + 'gunluk_istasyon_yildirim.pkl')

print("Veriler kaydedildi!")
print(f"\nAna veri seti: {ana_df.shape}")
print(f"Sütunlar: {ana_df.columns.tolist()}")

---
**Sonraki Adım:** `03_ozellik_muhendisligi.ipynb` - Özellik Mühendisliği